# 03 - Embeddings
Generate Titan Embeddings V2 (1024 dimensions) for graph nodes and enable vector similarity search on Neptune Analytics.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import time
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
GRAPH_ID = config["NEPTUNE_GRAPH_ID"]
EMBEDDING_MODEL = config["EMBEDDING_MODEL_ID"]
EMBEDDING_DIM = config["EMBEDDING_DIMENSIONS"]

if not GRAPH_ID:
    raise RuntimeError("NEPTUNE_GRAPH_ID not set. Run 02-neptune-graph.ipynb first.")

neptune = boto3.client("neptune-graph", region_name=REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)
tracker = ResourceTracker()

print(f"\u2713 Config loaded (graph={GRAPH_ID}, model={EMBEDDING_MODEL})")

## Step 2: Define Embedding Helper

Create a reusable function that calls Titan Embeddings V2 with 1024 dimensions.

In [ ]:
def get_embedding(text: str) -> list:
    """Generate a 1024-dim embedding using Titan Embeddings V2."""
    response = bedrock_runtime.invoke_model(
        modelId=EMBEDDING_MODEL,
        body=json.dumps({
            "inputText": text,
            "dimensions": EMBEDDING_DIM,
            "normalize": True
        })
    )
    result = json.loads(response["body"].read())
    return result["embedding"]

# Verify
test_emb = get_embedding("test")
print(f"\u2713 Embedding function ready (dim={len(test_emb)})")

## Step 3: Define Neptune Query Helper

In [ ]:
def run_query(query, parameters=None):
    """Execute an openCypher query against Neptune Analytics."""
    kwargs = {
        "graphIdentifier": GRAPH_ID,
        "queryString": query,
        "language": "OPEN_CYPHER"
    }
    if parameters:
        kwargs["parameters"] = parameters
    response = neptune.execute_query(**kwargs)
    return json.loads(response["payload"].read())

print("\u2713 Query helper defined")

## Step 4: Generate Embeddings for People

Create text representations of each person's profile and compute embeddings.

In [ ]:
# Fetch all people and their expertise
people_query = """
MATCH (p:Person)
OPTIONAL MATCH (p)-[r:KNOWS_ABOUT]->(t:Topic)
RETURN p.email AS email, p.name AS name, p.title AS title,
       collect(t.name) AS topics
"""
people_result = run_query(people_query)

print(f"\u231b Generating embeddings for {len(people_result['results'])} people...")
for person in people_result["results"]:
    # Build rich text representation
    topics_str = ", ".join(person["topics"]) if person["topics"] else "no listed topics"
    text = f"{person['name']}, {person['title']}. Expert in: {topics_str}"
    
    # Generate embedding
    embedding = get_embedding(text)
    
    # Store on Neptune node using vector property
    store_query = """
    MATCH (p:Person {email: $email})
    SET p.embedding = $embedding
    RETURN p.name AS name
    """
    run_query(store_query, parameters={
        "email": {"value": person["email"]},
        "embedding": {"value": embedding}
    })
    print(f"  \u2713 Embedded: {person['name']}")

print(f"\u2713 All person embeddings stored")

## Step 5: Generate Embeddings for Documents

In [ ]:
# Fetch all documents with their topics
docs_query = """
MATCH (d:Document)
OPTIONAL MATCH (d)-[:RELATED_TO]->(t:Topic)
OPTIONAL MATCH (p:Person)-[:AUTHORED]->(d)
RETURN d.doc_id AS doc_id, d.title AS title,
       collect(DISTINCT t.name) AS topics,
       collect(DISTINCT p.name) AS authors
"""
docs_result = run_query(docs_query)

print(f"\u231b Generating embeddings for {len(docs_result['results'])} documents...")
for doc in docs_result["results"]:
    topics_str = ", ".join(doc["topics"]) if doc["topics"] else ""
    authors_str = ", ".join(doc["authors"]) if doc["authors"] else ""
    text = f"{doc['title']}. Topics: {topics_str}. Authors: {authors_str}"
    
    embedding = get_embedding(text)
    
    store_query = """
    MATCH (d:Document {doc_id: $doc_id})
    SET d.embedding = $embedding
    RETURN d.title AS title
    """
    run_query(store_query, parameters={
        "doc_id": {"value": doc["doc_id"]},
        "embedding": {"value": embedding}
    })
    print(f"  \u2713 Embedded: {doc['title']}")

print(f"\u2713 All document embeddings stored")

## Step 6: Generate Embeddings for Topics

In [ ]:
topics_query = """
MATCH (t:Topic)
RETURN t.name AS name, t.category AS category
"""
topics_result = run_query(topics_query)

print(f"\u231b Generating embeddings for {len(topics_result['results'])} topics...")
for topic in topics_result["results"]:
    text = f"{topic['name']} - a {topic['category']} domain expertise area"
    embedding = get_embedding(text)
    
    store_query = """
    MATCH (t:Topic {name: $name})
    SET t.embedding = $embedding
    RETURN t.name AS name
    """
    run_query(store_query, parameters={
        "name": {"value": topic["name"]},
        "embedding": {"value": embedding}
    })
    print(f"  \u2713 Embedded: {topic['name']}")

print(f"\u2713 All topic embeddings stored")

## Step 7: Vector Similarity Search - Find Similar People

Use Neptune Analytics vector search to find people similar to a query.

In [ ]:
# Search for people related to a natural language query
search_text = "Who has deep expertise in container orchestration and deployment?"
query_embedding = get_embedding(search_text)

# Neptune Analytics vector search using CALL neptune.algo.vectors.get
vector_query = """
CALL neptune.algo.vectors.get(
    $embedding,
    {topK: 3, concurrency: 4}
)
YIELD node, score
WHERE 'Person' IN labels(node)
RETURN node.name AS name, node.title AS title, score
ORDER BY score DESC
"""

results = run_query(vector_query, parameters={
    "embedding": {"value": query_embedding}
})

print(f"Query: '{search_text}'")
print(f"\nTop matches:")
for row in results["results"]:
    print(f"  {row['name']} ({row['title']}) - similarity: {row['score']:.4f}")

print("\n\u2713 Vector search working")

## Step 8: Vector Search - Find Related Documents

In [ ]:
# Search for documents related to a query
doc_search_text = "How to set up automated deployment pipelines"
doc_query_embedding = get_embedding(doc_search_text)

doc_vector_query = """
CALL neptune.algo.vectors.get(
    $embedding,
    {topK: 3, concurrency: 4}
)
YIELD node, score
WHERE 'Document' IN labels(node)
RETURN node.title AS title, node.doc_id AS doc_id, score
ORDER BY score DESC
"""

results = run_query(doc_vector_query, parameters={
    "embedding": {"value": doc_query_embedding}
})

print(f"Query: '{doc_search_text}'")
print(f"\nTop document matches:")
for row in results["results"]:
    print(f"  {row['title']} ({row['doc_id']}) - similarity: {row['score']:.4f}")

print("\n\u2713 Document vector search working")

## Step 9: Combined Graph + Vector Query

Combine vector similarity with graph traversal for rich results.

In [ ]:
# Find similar people AND their authored documents
combined_text = "security and identity management"
combined_embedding = get_embedding(combined_text)

combined_query = """
CALL neptune.algo.vectors.get(
    $embedding,
    {topK: 3, concurrency: 4}
)
YIELD node, score
WHERE 'Person' IN labels(node)
WITH node AS person, score
OPTIONAL MATCH (person)-[:AUTHORED]->(d:Document)
RETURN person.name AS expert, score,
       collect(d.title) AS authored_docs
ORDER BY score DESC
"""

results = run_query(combined_query, parameters={
    "embedding": {"value": combined_embedding}
})

print(f"Query: '{combined_text}'")
print(f"\nExperts with their documents:")
for row in results["results"]:
    docs = ", ".join(row["authored_docs"]) if row["authored_docs"] else "(none)"
    print(f"  {row['expert']} (score: {row['score']:.4f})")
    print(f"    Documents: {docs}")

print("\n\u2713 Combined graph + vector query working")

## Step 10: Verify Embedding Coverage

In [ ]:
# Check how many nodes have embeddings
coverage_query = """
MATCH (n)
WHERE n.embedding IS NOT NULL
RETURN labels(n)[0] AS label, count(n) AS embedded_count
ORDER BY label
"""
result = run_query(coverage_query)

print("Embedding coverage:")
for row in result["results"]:
    print(f"  {row['label']}: {row['embedded_count']} nodes embedded")

print("\n\u2713 Embedding coverage verified")

## Summary

- [x] Titan Embeddings V2 generating 1024-dimension vectors
- [x] Person nodes embedded with profile + expertise text
- [x] Document nodes embedded with title + topic + author text
- [x] Topic nodes embedded with name + category
- [x] Vector similarity search working (person and document)
- [x] Combined graph traversal + vector search demonstrated

**Next:** [04-knowledge-base.ipynb](04-knowledge-base.ipynb) — Create OpenSearch Serverless collection and Bedrock Knowledge Base for RAG